In [17]:
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from transformers import TrainingArguments
from transformers import Trainer
from torch.optim import AdamW
from sklearn.model_selection import train_test_split

In [18]:
MODEL_NAME = "cahya/bert-base-indonesian-1.5G"

MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
df = pd.read_csv("rag_labeled.csv")
df["comment"] = df["comment"].fillna("").astype(str)
TAM_LABELS = ["PE","EE","SI","FC"]

train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED)

In [20]:
train_df[TAM_LABELS] = train_df[TAM_LABELS].astype(float)
test_df[TAM_LABELS] = test_df[TAM_LABELS].astype(float)
print(train_df.dtypes)

comment        str
PE         float64
EE         float64
SI         float64
FC         float64
dtype: object


In [21]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [22]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [23]:
def tokenize(batch):

    tokenized = tokenizer(
        batch["comment"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokenized["labels"] = [
        [pe, ee, si, fc]
        for pe, ee, si, fc in zip(
            batch["PE"],
            batch["EE"],
            batch["SI"],
            batch["FC"]
        )
    ]

    return tokenized

In [24]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 400/400 [00:00<00:00, 11427.06 examples/s]


In [25]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","labels"]
)

In [26]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    problem_type="regression",
    use_safetensors=True
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 405.30it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: cahya/bert-base-indonesian-1.5G
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing f

In [27]:
training_args = TrainingArguments(
    output_dir="models/sentiment_indobert",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",   # ganti dari evaluation_strategy
    logging_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01
)

In [28]:
from sklearn.metrics import mean_squared_error

def compute_metrics(eval_pred):

    preds, labels = eval_pred

    mse = mean_squared_error(labels, preds)

    rmse = np.sqrt(mse)

    return {
        "mse": mse,
        "rmse": rmse
    }

In [29]:
from sklearn.metrics import mean_squared_error

def compute_metrics(eval_pred):

    preds, labels = eval_pred

    mse = mean_squared_error(labels, preds)

    rmse = np.sqrt(mse)

    return {
        "mse": mse,
        "rmse": rmse
    }

In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [31]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss,Mse,Rmse
1,0.412636,0.321826,0.321826,0.567297
2,0.263264,0.299709,0.299709,0.547456
3,0.178121,0.317754,0.317754,0.563697


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.61it/s]


TrainOutput(global_step=300, training_loss=0.28467358271280924, metrics={'train_runtime': 126.6537, 'train_samples_per_second': 37.899, 'train_steps_per_second': 2.369, 'total_flos': 315738936115200.0, 'train_loss': 0.28467358271280924, 'epoch': 3.0})

In [32]:
trainer.save_model("models/utaut_indoroberta")
tokenizer.save_pretrained("models/utaut_indoroberta")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.14s/it]


('models/utaut_indoroberta\\tokenizer_config.json',
 'models/utaut_indoroberta\\tokenizer.json')